# 05 — Behavior scores and the map onto emotion

This notebook records how the playback workbook was scored, and how those scores were turned into valence, arousal, and social engagement. It describes the sheet as it was filled. It does not rescore the videos, and it is not the pose model in notebook 11.

**Data:** `data/playback_data_combined.xlsx`.

- **Sheet1** holds one row per playback. The sender (the bird whose call was used as the stimulus) has movement intensity (MI), body posture (BP), and directionality (D), plus valence (V), arousal (A), and social engagement (S). Up to three receivers have their own MI, BP, and D. Receivers were not given V, A, or S columns.
- **Sheet2** is the signed observation key.
- **Sheet3** is the shorter positive-only list.

## The observation key

A dimension score is the sum of the points for the items that were checked. Each item on the key is worth +1 or −1.

**Sheet2** is the key that can produce both signs.

- **Movement intensity.** Positive means intense movement: long distance, quick movement, many hops, many flights, a long bout of quick movement. Negative means little movement: short distance, slow or no movement, few hops, few flights, a short bout.
- **Body posture.** Positive means excited: wings spread, body stretched, body upright, eyes wide open, feathers sleeked. Negative means aggressive: feathers fluffed, body hunched, mouth open, tail stiff, wings folded.
- **Directionality.** Facing, approaching, or holding on another bird is negative. The same actions toward the speaker (the human or the stimulus) are positive. Each side lists orientation, a short latency, a long duration, an approach, and few switches away.

Some pairs are opposites (long versus short, quick versus still), so a bird was not given both ends of a pair. The totals that were entered are small integers, about −5 to +5.

**Sheet3** keeps only +1 items. Direction is only toward the speaker. It adds a short call latency, and it lists hop, flight, and quick wing-flapping as present-or-not. It has no −1 items, so it cannot by itself produce the negative MI, BP, or D values on Sheet1. Those come from Sheet2.

The notes in the first column of each sheet ("Positive: …", "Negative: …") sit beside the item list. The point that was added is the Score column.

On Sheet1 the sender scores use both signs, so they follow Sheet2. Every receiver MI, BP, and D in the workbook is 0 or positive, the same sign pattern as Sheet3. Receivers were not given valence, arousal, or social engagement.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA = Path(r"D:\CHIRP\data\playback_data_combined.xlsx")
DIMENSIONS = ("Movement Intensity", "Body Posture", "Directionality")


def load_key(sheet_name: str) -> pd.DataFrame:
    """Sheet2 and Sheet3: a title row, then Dimension / Metric / Score."""
    raw = pd.read_excel(DATA, sheet_name=sheet_name, header=None)
    body = raw.iloc[2:].copy()
    body.columns = ["Dimension", "Metric", "Score"]
    return body.reset_index(drop=True)


def points_table(key: pd.DataFrame) -> pd.DataFrame:
    """One row per scored item. Polarity notes stay on the raw sheet, not in this table."""
    dimension = None
    rows = []
    for _, row in key.iterrows():
        label = row["Dimension"]
        if isinstance(label, str) and label in DIMENSIONS:
            dimension = label
        metric = row["Metric"]
        if isinstance(metric, str) and metric.strip():
            rows.append({"Dimension": dimension, "Metric": metric, "Points": int(row["Score"])})
    return pd.DataFrame(rows)


sheet1 = pd.read_excel(DATA, sheet_name="Sheet1")
sheet2 = load_key("Sheet2")
sheet3 = load_key("Sheet3")

def as_stored(key: pd.DataFrame) -> pd.DataFrame:
    shown = key.astype(object)
    return shown.where(shown.notna(), "")


print("Sheet2, as stored")
display(as_stored(sheet2))
print("Sheet2 items")
display(points_table(sheet2))
print("Sheet3, as stored")
display(as_stored(sheet3))
print("Sheet3 items")
display(points_table(sheet3))

Sheet2, as stored


,Dimension,Metric,Score
0,Movement Intensity,Movement Distance Long,1
1,Positive: Intense Movement,Movement Quick,1
2,Negative: Little Movement,Number of Hops Large,1
3,,Number of Flights Large,1
4,,Duration of Quick Movement Long,1
5,,Movement Distance Short,-1
6,,Movement Slow/No Movement,-1
7,,Number of Hops Small,-1
8,,Number of Flights Small,-1
9,,Duration of Quick Movement Short,-1


Sheet2 items


,Dimension,Metric,Points
0,Movement Intensity,Movement Distance Long,1
1,Movement Intensity,Movement Quick,1
2,Movement Intensity,Number of Hops Large,1
3,Movement Intensity,Number of Flights Large,1
4,Movement Intensity,Duration of Quick Movement Long,1
5,Movement Intensity,Movement Distance Short,-1
6,Movement Intensity,Movement Slow/No Movement,-1
7,Movement Intensity,Number of Hops Small,-1
8,Movement Intensity,Number of Flights Small,-1
9,Movement Intensity,Duration of Quick Movement Short,-1


Sheet3, as stored


,Dimension,Metric,Score
0,Movement Intensity,Movement Distance Long,1
1,,Movement Quick,1
2,,Hop,1
3,,Flight,1
4,,Quick Flapping of Wings,1
5,Body Posture,Wings Spread,1
6,,Body Stretched,1
7,,Body Upright,1
8,,Eyes Wide Open,1
9,,Feather Sleeked,1


Sheet3 items


,Dimension,Metric,Points
0,Movement Intensity,Movement Distance Long,1
1,Movement Intensity,Movement Quick,1
2,Movement Intensity,Hop,1
3,Movement Intensity,Flight,1
4,Movement Intensity,Quick Flapping of Wings,1
5,Body Posture,Wings Spread,1
6,Body Posture,Body Stretched,1
7,Body Posture,Body Upright,1
8,Body Posture,Eyes Wide Open,1
9,Body Posture,Feather Sleeked,1


## From the three scores to emotion

Sender V, A, and S on Sheet1 are weighted sums of that row's sender MI, BP, and D. There is no intercept. Checked on every sender row that has all three scores, the largest absolute difference from the stored column is numerical noise.

$$
\begin{aligned}
V &= 0.1\,\mathrm{MI} + 0.5\,\mathrm{BP} + 0.3\,D \\
A &= 0.4\,|\mathrm{MI}| + 0.6\,|\mathrm{BP}| \\
S &= 0.4\,\mathrm{BP} + 0.6\,D
\end{aligned}
$$

Read against the key:

- **Valence** follows posture first (excited versus aggressive), then direction (toward the stimulus versus toward another bird), and only a little of the signed movement score.
- **Arousal** uses the size of the movement score and the size of the posture score. The sign is dropped, and direction is not in the sum. A strongly marked posture raises arousal whether it was coded excited or aggressive.
- **Social engagement** follows direction first, then posture. Movement intensity is not in this sum.

Inside the scores that were entered, these sums already land on the stored ranges. Nothing further was clipped.

These are the sums in the workbook. A later design note sketched a different, nonlinear map from the same three scores. That note does not describe how these columns were filled.

In [2]:
mi = sheet1["SenderMI"]
bp = sheet1["SenderBP"]
direction = sheet1["SenderD"]
complete = mi.notna() & bp.notna() & direction.notna()

valence = 0.1 * mi + 0.5 * bp + 0.3 * direction
arousal = 0.4 * mi.abs() + 0.6 * bp.abs()
social = 0.4 * bp + 0.6 * direction

check = pd.DataFrame(
    {
        "V": (valence - sheet1["SenderV"]).abs(),
        "A": (arousal - sheet1["SenderA"]).abs(),
        "S": (social - sheet1["SenderS"]).abs(),
    }
)
print(f"complete sender rows: {int(complete.sum())}")
print("max absolute difference")
display(check.loc[complete].max().to_frame("max_abs"))

example = sheet1.loc[complete, ["FileName", "SenderMI", "SenderBP", "SenderD", "SenderV", "SenderA", "SenderS"]].head(6).copy()
example["V_sum"] = valence.loc[example.index]
example["A_sum"] = arousal.loc[example.index]
example["S_sum"] = social.loc[example.index]
print("stored columns beside the sums")
display(example)

print("sender score ranges")
display(sheet1.loc[complete, ["SenderMI", "SenderBP", "SenderD", "SenderV", "SenderA", "SenderS"]].agg(["min", "max"]))

receiver_cols = [f"Receiver{i}{metric}" for i in (1, 2, 3) for metric in ("MI", "BP", "D")]
print("receivers have MI, BP, and D only; in this sheet those scores are all >= 0")
display(sheet1[receiver_cols].agg(["min", "max"]))

complete sender rows: 316
max absolute difference


,max_abs
V,4.440892e-16
A,4.440892e-16
S,4.440892e-16


stored columns beside the sums


,FileName,SenderMI,SenderBP,SenderD,SenderV,SenderA,SenderS,V_sum,A_sum,S_sum
0,Aster_alarmrespondingtohuman_1524,4,-2,-4,-1.8,2.8,-3.2,-1.8,2.8,-3.2
1,Aster_alertoncurtain_1906,2,1,-2,0.1,1.4,-0.8,0.1,1.4,-0.8
2,Aster_aloneresponding_1458,1,0,0,0.1,0.4,0.0,0.1,0.4,0.0
3,Aster_alonetalking_1335,1,1,0,0.6,1.0,0.4,0.6,1.0,0.4
4,Aster_angryathomeoccupied_1904,4,-4,3,-0.7,4.0,0.2,-0.7,4.0,0.2
5,Aster_announcearrival_1342,3,3,-1,1.5,3.0,0.6,1.5,3.0,0.6


sender score ranges


,SenderMI,SenderBP,SenderD,SenderV,SenderA,SenderS
min,-4,-5,-5,-3.1,0.0,-4.6
max,5,5,5,4.4,5.0,5.0


receivers have MI, BP, and D only; in this sheet those scores are all >= 0


,Receiver1MI,Receiver1BP,Receiver1D,Receiver2MI,Receiver2BP,Receiver2D,Receiver3MI,Receiver3BP,Receiver3D
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,4.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0


## Z-scores used later, for comparison

The consistency notebooks compare receivers on one scale. After the scores above were entered, each metric was z-scored with the population standard deviation (divide by $n$, not $n-1$).

- Receiver MI, receiver BP, and receiver D each have one scaler. The three receiver columns for that metric are stacked, and that mean and standard deviation are used for all three.
- Sender MI, BP, D, V, A, and S are each scaled on their own column.

The raw scores were left on Sheet1. The scaled copies are the `_z` columns, and the means and standard deviations are on the `zscore_params` sheet. Emotion itself was computed from the raw integer scores, not from the z-scores.

In [3]:
params = pd.read_excel(DATA, sheet_name="zscore_params")
behavior_params = params[params["column"].astype(str).str.contains(r"MI$|BP$|D$|Sender[VASC]", regex=True)].copy()
display(behavior_params.reset_index(drop=True))

,column,z_column,group,mean,std,n
0,SenderMI,SenderMI_z,sender_independent,2.683544,1.280766,316
1,SenderBP,SenderBP_z,sender_independent,1.873418,2.876650,316
2,SenderD,SenderD_z,sender_independent,2.183544,3.338192,316
3,SenderV,SenderV_z,sender_independent,1.860127,1.792369,316
4,SenderA,SenderA_z,sender_independent,3.005063,1.055414,316
5,SenderS,SenderS_z,sender_independent,2.059494,2.370388,316
6,Receiver1MI,Receiver1MI_z,receiver_pooled_MI,1.250811,1.063300,925
7,Receiver2MI,Receiver2MI_z,receiver_pooled_MI,1.250811,1.063300,925
8,Receiver3MI,Receiver3MI_z,receiver_pooled_MI,1.250811,1.063300,925
9,Receiver1BP,Receiver1BP_z,receiver_pooled_BP,2.243243,1.026287,925
